In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
from pathlib import Path
import math
import pickle
import random
import sys
from copy import deepcopy
from datetime import datetime
from itertools import product

import pandas as pd

# Funktioniert unabhängig davon, ob der Kernel im Projektordner oder in SimRepair/ startet.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "SimRepair":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "SimRepair").is_dir():
    raise RuntimeError(
        "Bitte starte das Notebook aus dem SCOPE-Projektordner oder aus SCOPE/SimRepair."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Unterstützt auch die älteren, unqualifizierten Imports in späteren Guide-Zellen.
if str(PROJECT_ROOT / "SimRepair") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "SimRepair"))

from SimRepair import confounding_level, simulation
from SimRepair.activity_execution import ActivityExecutioner

OUTPUT_DIR = PROJECT_ROOT / "data" / "SimRepair"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Lokale, leichtgewichtige Speicherung: kein Import des gesamten Modell-/Torch-Stacks nötig.
def save_data(data, file_path):
    file_path = Path(file_path).with_suffix(".pkl")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with file_path.open("wb") as file:
        pickle.dump(data, file)
    return file_path


def load_data(file_path):
    file_path = Path(file_path).with_suffix(".pkl")
    with file_path.open("rb") as file:
        return pickle.load(file)

print(f"Projektordner: {PROJECT_ROOT}")
print(f"Ausgabeordner: {OUTPUT_DIR}")

Projektordner: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/My_Workspace/SCOPE_WORKSPACE
Ausgabeordner: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/My_Workspace/SCOPE_WORKSPACE/data/SimRepair


In [3]:
# DATASET parameters
dataset_params = {}

# Kleine Standardwerte, damit das Guide-Notebook schnell ausprobiert werden kann.
# Für vollständige Experimente diese Werte z. B. wieder auf 10_000 setzen.
dataset_params["train_size"] = 10
dataset_params["test_size"] = 10
dataset_params["val_share"] = 0.5
dataset_params["train_val_size"] = 10
dataset_params["test_val_size"] = min(
    int(dataset_params["val_share"] * dataset_params["test_size"]), 1000
)
dataset_params["simulation_start"] = datetime(2024, 3, 20, 8, 0)
dataset_params["random_seed_train"] = 82 * 82
dataset_params["random_seed_test"] = 130 * 130

# Process columns
dataset_params["log_cols"] = [
    "case_nr", "activity", "timestamp", "elapsed_time", "duration",
    "bike_value", "repair_severity", "quality_uncertainty", "process_type",
    "employee_competence", "material_quality", "customer_patience",
    "customer_friendliness", "outcome",
]
dataset_params["case_cols"] = ["bike_value", "repair_severity"]
dataset_params["event_cols"] = [
    "activity", "elapsed_time", "quality_uncertainty", "process_type",
    "employee_competence", "material_quality",
]
dataset_params["cat_cols"] = ["activity", "process_type"]
dataset_params["scale_cols"] = [
    "elapsed_time", "bike_value", "repair_severity", "quality_uncertainty",
    "employee_competence", "material_quality", "outcome",
]
dataset_params["last_state_cols"] = ["elapsed_time"]

# Intervention
dataset_params["intervention_info"] = {}
dataset_params["intervention_info"]["name"] = ["choose_procedure"]
# Alternatives: ["conduct_qc"] or ["choose_procedure", "conduct_qc"]

if dataset_params["intervention_info"]["name"] == ["choose_procedure"]:
    dataset_params["intervention_info"]["data_impact"] = ["direct"]
    dataset_params["intervention_info"]["actions"] = [["start_standard", "start_priority"]]
    dataset_params["intervention_info"]["action_width"] = [2]
    dataset_params["intervention_info"]["action_depth"] = [1]
    dataset_params["intervention_info"]["activities"] = [["start_standard", "start_priority"]]
    dataset_params["intervention_info"]["column"] = ["activity"]
    dataset_params["intervention_info"]["start_control_activity"] = [["initiate_case"]]
    dataset_params["intervention_info"]["end_control_activity"] = [["initiate_case"]]
elif dataset_params["intervention_info"]["name"] == ["conduct_qc"]:
    dataset_params["intervention_info"]["data_impact"] = ["direct"]
    dataset_params["intervention_info"]["actions"] = [["shipping", "conduct_qc"]]
    dataset_params["intervention_info"]["action_width"] = [2]
    dataset_params["intervention_info"]["action_depth"] = [4]
    dataset_params["intervention_info"]["activities"] = [["shipping", "conduct_qc"]]
    dataset_params["intervention_info"]["column"] = ["activity"]
    dataset_params["intervention_info"]["start_control_activity"] = [["repair_priority", "repair_standard", "improve_rework"]]
    dataset_params["intervention_info"]["end_control_activity"] = [["repair_priority", "repair_standard", "improve_rework"]]
elif dataset_params["intervention_info"]["name"] == ["choose_procedure", "conduct_qc"]:
    dataset_params["intervention_info"]["data_impact"] = ["direct", "direct"]
    dataset_params["intervention_info"]["actions"] = [
        ["start_standard", "start_priority"], ["shipping", "conduct_qc"]
    ]
    dataset_params["intervention_info"]["action_width"] = [2, 2]
    dataset_params["intervention_info"]["action_depth"] = [1, 4]
    dataset_params["intervention_info"]["activities"] = [
        ["start_standard", "start_priority"], ["shipping", "conduct_qc"]
    ]
    dataset_params["intervention_info"]["column"] = ["activity", "activity"]
    dataset_params["intervention_info"]["start_control_activity"] = [
        ["initiate_case"], ["repair_priority", "repair_standard", "improve_rework"]
    ]
    dataset_params["intervention_info"]["end_control_activity"] = [
        ["initiate_case"], ["repair_priority", "repair_standard", "improve_rework"]
    ]

dataset_params["intervention_info"]["retain_method"] = "precise"
dataset_params["intervention_info"]["action_combinations"] = list(
    product(*dataset_params["intervention_info"]["actions"])
)
dataset_params["intervention_info"]["action_width_combinations"] = math.prod(
    dataset_params["intervention_info"]["action_width"]
)
dataset_params["intervention_info"]["action_depth_combinations"] = math.prod(
    dataset_params["intervention_info"]["action_depth"]
)
dataset_params["intervention_info"]["len"] = [
    width if width > 2 else 1
    for width in dataset_params["intervention_info"]["action_width"]
]
dataset_params["intervention_info"]["RCT"] = False
dataset_params["filename"] = "repair_log_" + str(dataset_params["intervention_info"]["name"])

# Workshop policies
dataset_params["policies_info"] = {
    "general": "real",
    "take_employee": {"max_employee_waits": 3},
    "conduct_qc": {"max_qc": 3},
    "max_rerepairs": 3,
}

## Offline Mode

### - Use to generate offline, fixed datasets for methods requiring offline training

In [4]:
# Initiate simulation
offline_gen_normal = simulation.PresProcessGenerator(
    dataset_params, dataset_params["random_seed_train"]
)

# Generate training and validation data using the bank policy
train_normal = offline_gen_normal.run_simulation_normal(dataset_params["train_size"])
train_normal_val = offline_gen_normal.run_simulation_normal(
    dataset_params["train_val_size"], seed_to_add=88
)

print(
    f"Trainingslog: {train_normal['case_nr'].nunique()} Fälle, "
    f"{len(train_normal)} Events"
)
display(train_normal.head(20))

# Einen vollständigen Prozessfall ansehen
first_case_nr = train_normal["case_nr"].iloc[0]
display(train_normal.loc[train_normal["case_nr"] == first_case_nr])

Trainingslog: 10 Fälle, 118 Events


,case_nr,activity,bike_value,repair_severity,quality_uncertainty,customer_patience,customer_friendliness,outcome,duration,timestamp,elapsed_time,process_type,employee_competence,material_quality
0,0,initiate_case,6,3,0.559620,3,4,0,86400.000000,2024-03-20 08:00:00.000000,0.000000,NaN,NaN,NaN
1,0,start_priority,6,3,0.559620,3,4,0,86400.000000,2024-03-21 08:00:00.000000,1.000000,priority,NaN,NaN
2,0,wait_for_employee,6,3,0.559620,3,4,0,69120.000000,2024-03-22 08:00:00.000000,2.000000,priority,5.0,NaN
3,0,wait_for_employee,6,3,0.559620,3,4,0,69120.000000,2024-03-23 03:12:00.000000,2.800000,priority,5.0,NaN
4,0,wait_for_employee,6,3,0.559620,3,4,0,69120.000000,2024-03-23 22:24:00.000000,3.600000,priority,4.0,NaN
5,0,use_spare_parts,6,3,0.559620,3,4,0,38880.000000,2024-03-24 17:36:00.000000,4.400000,priority,4.0,5.0
6,0,repair_priority,6,3,0.559620,3,4,0,293142.857143,2024-03-25 04:24:00.000000,4.850000,priority,4.0,5.0
7,0,conduct_qc,6,3,0.223848,3,4,0,172800.000000,2024-03-28 13:49:42.857143,8.242857,priority,4.0,5.0
8,0,improve_rework,6,3,0.223848,3,4,0,259200.000000,2024-03-30 13:49:42.857143,10.242857,priority,4.0,5.0
9,0,shipping,6,3,0.223848,3,4,0,86400.000000,2024-04-02 13:49:42.857143,13.242857,priority,4.0,5.0


,case_nr,activity,bike_value,repair_severity,quality_uncertainty,customer_patience,customer_friendliness,outcome,duration,timestamp,elapsed_time,process_type,employee_competence,material_quality
0,0,initiate_case,6,3,0.559620,3,4,0,86400.000000,2024-03-20 08:00:00.000000,0.000000,NaN,NaN,NaN
1,0,start_priority,6,3,0.559620,3,4,0,86400.000000,2024-03-21 08:00:00.000000,1.000000,priority,NaN,NaN
2,0,wait_for_employee,6,3,0.559620,3,4,0,69120.000000,2024-03-22 08:00:00.000000,2.000000,priority,5.0,NaN
3,0,wait_for_employee,6,3,0.559620,3,4,0,69120.000000,2024-03-23 03:12:00.000000,2.800000,priority,5.0,NaN
4,0,wait_for_employee,6,3,0.559620,3,4,0,69120.000000,2024-03-23 22:24:00.000000,3.600000,priority,4.0,NaN
5,0,use_spare_parts,6,3,0.559620,3,4,0,38880.000000,2024-03-24 17:36:00.000000,4.400000,priority,4.0,5.0
6,0,repair_priority,6,3,0.559620,3,4,0,293142.857143,2024-03-25 04:24:00.000000,4.850000,priority,4.0,5.0
7,0,conduct_qc,6,3,0.223848,3,4,0,172800.000000,2024-03-28 13:49:42.857143,8.242857,priority,4.0,5.0
8,0,improve_rework,6,3,0.223848,3,4,0,259200.000000,2024-03-30 13:49:42.857143,10.242857,priority,4.0,5.0
9,0,shipping,6,3,0.223848,3,4,0,86400.000000,2024-04-02 13:49:42.857143,13.242857,priority,4.0,5.0


In [ ]:
# Generate RCT data.
# Used to check if Bank policy is better than random interventions (RCT)
# Used to reduce confounding and check if model still learns good policies
# Training set for our Policy is generated via careful deviation from bank policy, not random interventions.
dataset_params_RCT = deepcopy(dataset_params)
dataset_params_RCT["intervention_info"]["RCT"] = True
dataset_params_RCT["random_seed_train"] = dataset_params["random_seed_train"]*10
dataset_params_RCT["simulation_start"] = deepcopy(offline_gen_normal.simulation_end)

# Initiate simulation
offline_gen_RCT = simulation.PresProcessGenerator(dataset_params_RCT, dataset_params_RCT["random_seed_train"])

# Generate training data
train_RCT = offline_gen_RCT.run_simulation_normal(dataset_params_RCT["train_size"])

# Generate validation data
train_RCT_val = offline_gen_RCT.run_simulation_normal(dataset_params_RCT["train_val_size"], seed_to_add=88)

### - Vary the confounding level for offline datasets

In [9]:
# Set confounding level delta (combination of training data generated by bank policy and RCT)
delta = 0.5
train = confounding_level.set_delta(data=train_normal, data_RCT=train_RCT, delta=delta)
train_val = confounding_level.set_delta(data=train_normal_val, data_RCT=train_RCT_val, delta=delta)

##### Save data

In [10]:
def save_dataset_bundle(suffix=""):
    """Speichert alle erzeugten Datensätze portabel unter data/SimBank/."""
    stem = f"{dataset_params['filename']}_{dataset_params['train_size']}"
    datasets = {
        "dataset_params": dataset_params,
        "train_normal": train_normal,
        "train_normal_val": train_normal_val,
        "train_RCT": train_RCT,
        "train_RCT_val": train_RCT_val,
    }
    saved_paths = []
    for name, data in datasets.items():
        suffix_part = f"_{suffix}" if suffix else ""
        saved_paths.append(
            save_data(data, OUTPUT_DIR / f"{stem}_{name}{suffix_part}")
        )
    return saved_paths


# Beispiel: Standard-Datensatz speichern und die erzeugten Pfade anzeigen
saved_paths = save_dataset_bundle()
saved_paths

[PosixPath("/Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/My_Workspace/SCOPE_WORKSPACE/data/SimRepair/repair_log_['choose_procedure']_10_dataset_params.pkl"),
 PosixPath("/Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/My_Workspace/SCOPE_WORKSPACE/data/SimRepair/repair_log_['choose_procedure']_10_train_normal.pkl"),
 PosixPath("/Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/My_Workspace/SCOPE_WORKSPACE/data/SimRepair/repair_log_['choose_procedure']_10_train_normal_val.pkl"),
 PosixPath("/Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/My_Workspace/SCOPE_WORKSPACE/data/SimRepair/repair_log_['choose_procedure']_10_train_RCT.pkl"),
 PosixPath("/Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/My_Workspace/SCOPE_WORKSPACE/data/SimRepair/repair_log_['choose_procedure']_10_train_RCT_val.pkl")]

In [44]:
# Optional: zusätzlich mit dem früher verwendeten Namenssuffix speichern.
# save_dataset_bundle("ProCause_training")

In [36]:
# Optional:
# save_dataset_bundle("ProCause_tuning")

In [8]:
# Optional:
# save_dataset_bundle("BOZORGI_RealCause")

## Online Mode

### - Use to generate cases (episodes) for methods requiring online training
### - Use to generate cases (episodes) for testing the performance of (both onffline and online) methods (make sure to specify the seed to generate the same cases for each method)

In [11]:
# Initiate simulation
online_gen = simulation.PresProcessGenerator(dataset_params, dataset_params["random_seed_test"])

# Start simulation
for case in range(dataset_params["test_size"]):
    prefix_list = online_gen.start_simulation_inference(seed_to_add=case)

    # Continue simulation and specify actions (for example using a model) until no more interventions are available
    while online_gen.int_points_available:
        
        # Specify the action to take
        index_action_taken = random.randrange(len(dataset_params["intervention_info"]["actions"][online_gen.current_int_index]))
        print("Action taken: ", dataset_params["intervention_info"]["actions"][online_gen.current_int_index][index_action_taken])

        # Continue simulation
        prefix_list = online_gen.continue_simulation_inference(index_action_taken)

    # End simulation
    full_case = online_gen.end_simulation_inference()
    print("\n", "Full case: ", full_case)

Action taken:  start_priority

 Full case:  [{'case_nr': 0, 'activity': 'initiate_case', 'bike_value': 1, 'repair_severity': 5, 'quality_uncertainty': 0.5423146261210254, 'customer_patience': 8, 'customer_friendliness': 6, 'outcome': 0, 'duration': 86400, 'timestamp': datetime.datetime(2024, 3, 20, 8, 0), 'elapsed_time': 0}, {'case_nr': 0, 'activity': 'start_priority', 'bike_value': 1, 'repair_severity': 5, 'quality_uncertainty': 0.5423146261210254, 'customer_patience': 8, 'customer_friendliness': 6, 'outcome': 0, 'process_type': 'priority', 'duration': 86400, 'timestamp': datetime.datetime(2024, 3, 21, 8, 0), 'elapsed_time': 1.0}, {'case_nr': 0, 'activity': 'wait_for_employee', 'bike_value': 1, 'repair_severity': 5, 'quality_uncertainty': 0.5423146261210254, 'customer_patience': 8, 'customer_friendliness': 6, 'process_type': 'priority', 'outcome': 0, 'employee_competence': 5, 'duration': 69120.0, 'timestamp': datetime.datetime(2024, 3, 22, 8, 0), 'elapsed_time': 2.0}, {'case_nr': 0, '

### - Example of bank policy performance: the first cell includes the bank policy rules, the second cell contains the performance calculation

In [ ]:
# Bank policy

from activity_execution import ActivityExecutioner
# Welche Aktion würde die fest programmierte normale Bankpolitik wählen?
# Das hat nicht mit der besten Aktio zu tun sondern ist nur die Aktion, die die Bank wählen würde
def get_bank_best_action(prefix_list, current_int_index):
        prefix_without_int = prefix_list[0][0:-1]
        prev_event = prefix_without_int[-1]
        action_index = 0
        
        if dataset_params["intervention_info"]["name"][current_int_index] == "time_contact_HQ":
            cancel_condition = ((prev_event["unc_quality"] == 0 and prev_event["est_quality"] < dataset_params["policies_info"]["min_quality"] and prev_event["noc"] >= dataset_params["policies_info"]["max_noc"]) or (prev_event["noc"] >= dataset_params["policies_info"]["max_noc"] and prev_event["unc_quality"] > 0))
            contact_condition = (prev_event["noc"] < 2 and prev_event["unc_quality"] == 0 and prev_event["amount"] > 10000 and prev_event["est_quality"] >= dataset_params["policies_info"]["min_quality"])

            if cancel_condition:
                action_index = 0
            elif contact_condition:
                action_index = 1
        
        elif dataset_params["intervention_info"]["name"][current_int_index] == "choose_procedure":
            priority_condition = (prev_event["amount"] > dataset_params["policies_info"]["choose_procedure"]["amount"] and prev_event["est_quality"] >= dataset_params["policies_info"]["choose_procedure"]["est_quality"])

            if priority_condition:
                action_index = 1
            else:
                action_index = 0
        
        elif dataset_params["intervention_info"]["name"][current_int_index] == "set_ir_3_levels":
            activity_executioner = ActivityExecutioner()
            ir, _, _ = activity_executioner.calculate_offer(prev_event=prev_event, intervention_info=dataset_params["intervention_info"])
            action_index = dataset_params["intervention_info"]["actions"][current_int_index].index(ir)
        
        return action_index

In [ ]:
# GANZ TOP LEVEL DIE SIMULATION
# Nutzt dieselbe kleine Fallzahl wie die übrigen Guide-Beispiele.
n_cases = dataset_params["test_size"]

# Init performance metrics
bank_performance = 0

#Init data generator
case_gen = simulation.PresProcessGenerator(dataset_params, seed=dataset_params["random_seed_test"])

#Run
for case_nr in range(n_cases):
    prefix_list = []
    prefix_list = case_gen.start_simulation_inference(seed_to_add=case_nr)
    while case_gen.int_points_available:
        bank_best_action = get_bank_best_action(prefix_list, case_gen.current_int_index)
        # Break if intervention done or in last timing
        prefix_list = case_gen.continue_simulation_inference(bank_best_action)

    full_case = case_gen.end_simulation_inference()
    full_case = pd.DataFrame(full_case)
    bank_performance += full_case["outcome"].iloc[-1]

print("Bank performance: ", bank_performance)

Bank performance:  29976509.22284894


### - Example of random policy performance: random policy includes sampling a random timing of the intervention (if action depth > 1) and sampling a random intervention action

In [10]:
# Specify number of cases and iteration.
# Für die Paper-Ergebnisse: test_size auf 10_000 setzen und Iterationen 0 bis 4 mitteln.
n_cases = dataset_params["test_size"]
iteration = 0

# Init performance metrics
random_performance = 0

# Init data generator
case_gen = simulation.PresProcessGenerator(dataset_params, seed=dataset_params["random_seed_test"])

random_object_for_random_policy = random.Random(dataset_params["random_seed_test"] + 5*iteration)

# Run
for case_nr in range(n_cases):
    prefix_list = []
    prefix_list = case_gen.start_simulation_inference(seed_to_add=case_nr)

    # Sample random timing if action depth is more than 1 (else, timing is just 0)
    timings = [0] * len(dataset_params["intervention_info"]["name"])
    random_best_timings = []
    for int_index in range(len(dataset_params["intervention_info"]["name"])):
        if dataset_params["intervention_info"]["name"][int_index] == "time_contact_HQ":
            random_best_timings.append(random_object_for_random_policy.choice(range(dataset_params["intervention_info"]["action_depth"][int_index])) * 2)
        else:
            random_best_timings.append(0)
    random_best_action = 0 # control

    while case_gen.int_points_available:
        
        # Only do an action if the current timing is the same as the randomly sampled timing
        if timings[case_gen.current_int_index] == random_best_timings[case_gen.current_int_index]:
            # Sample a random intervention action
            random_best_action = random_object_for_random_policy.choice(range(dataset_params["intervention_info"]["action_width"][case_gen.current_int_index]))
        timings[case_gen.current_int_index] += 1
        
        prefix_list = case_gen.continue_simulation_inference(random_best_action)

    full_case = case_gen.end_simulation_inference()
    full_case = pd.DataFrame(full_case)
    random_performance += full_case["outcome"].iloc[-1]

print("Random policy performance: ", random_performance)

Random policy performance:  23851100.968992893
